# Research Task - Local Transit Projects List #890
https://github.com/cal-itp/data-analyses/issues/890

Start working on a local (transit) projects list, perhaps by reaching out to people like Jeff Tumlin?
Amanda to start from LRTP lists, Christian to shadow/ask more detailed transit priority questions. "Across CA, how much would we save with transit priority implementation?"

Connected to Julian (student assistant) scope (analyze benefits from list)

Received more clarification. Starting with MPOs LRTP, identify projects that improve existing transit performance (upgrading something, improve ridership, etc). Then compare this short list to the remaining projects in LRTP (% of..., cost vs...). May need to look at previous LRTPs to see if same projects were mentioned previously to see if projects were completed or not.


## Next Steps
Start with reading in Amanda's data from gcs (list of all projects from MPO LRTPs)

In [1]:
import pandas as pd


In [6]:
my_df = pd.read_excel('gs://calitp-analytics-data/data-analyses/project_list/LRTP/all_LRTP_LOST.xlsx')

In [7]:
my_df.shape

(16276, 9)

In [8]:
# filtering out LOST projects
# df.loc[df.data_source != "Lost"] 
my_df = my_df.loc[my_df.data_source != 'Lost']

In [ ]:
#observe shape after removing lost projects
display(my_df.shape)


## Next
Find Amanda's code snippet of filtering the df by string values. specifically, filtering the projects that are trasnit 
related (either project name/desc/notes)

see these for reference:

https://github.com/cal-itp/data-analyses/blob/2b88d23f3895840cd8714f213ec6a4c1fc55c088/project_list/_categorizing_utils.py#L135

https://github.com/cal-itp/data-analyses/blob/main/project_list/_specific_list_utils.py#L44


In [10]:
transit_keywords = [
        "bus",
        "metro",
        "station",  # Station comes up a few times as a charging station and also as a train station
        "transit",
        "fare",
        "brt",
        "yarts",
        "railroad",
        "highway-rail",
        "streetcar",
        "mass transit",]
        # , 'station' in description and 'charging station' not in description

In [24]:
#test of isin() for transit keywords

test = my_df[my_df['project_title'].str.contains('|'.join(transit_keywords))]
test

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [15]:
#ripping code from Amanda
def lower_case(df, columns_to_search: list):
    """
    Lowercase and clean certain columns:
    """
    new_columns = []
    for i in columns_to_search:
        df[f"lower_case_{i}"] = (
            df[i]
            .str.lower()
            .fillna("none")
            .str.replace("-", "")
            .str.replace(".", "")
            .str.replace(":", "")
        )
        new_columns.append(f"lower_case_{i}")

    return df, new_columns

#another funct.
def find_keywords(df, columns_to_search: list, keywords_search: list):
    """
    Find keywords in certain columns
    """
    df2, lower_case_cols_list = lower_case(df, columns_to_search)

    keywords_search = f"({'|'.join(keywords_search)})"

    for i in lower_case_cols_list:
        df2[f"{i}_keyword_search"] = (
            df2[i].str.extract(keywords_search).fillna("keyword not found")
        )

    return df2

#another funct.
def filter_projects(
    df,
    columns_to_search: list,
    keywords_search: list,
    file_name: str,
    save_to_gcs: bool = True,
):

    # Filter out for Cordon
    df = find_keywords(df, columns_to_search, keywords_search)
    df2 = (
        df[
            (df.lower_case_project_title_keyword_search != "keyword not found")
            | (df.lower_case_project_description_keyword_search != "keyword not found")
        ]
    ).reset_index(drop=True)

    # Delete out non HOV projects that were accidentally picked up
    projects_to_delete = [
        "SR 17 Corridor Congestion Relief in Los Gatos",
        "Interstate 380 Congestion Improvements",
    ]
    df2 = df2[~df2.project_title.isin(projects_to_delete)].reset_index(drop=True)

    # Change cases
    for i in ["project_title", "project_description"]:
        df2[i] = df2[i].str.title()

    # Drop invalid geometries
    #gdf = df2[df2.geometry != None].reset_index(drop=True)
    #gdf = gdf.set_geometry("geometry")
    #gdf.geometry = gdf.geometry.set_crs("EPSG:4326")
    #gdf = gdf[gdf.geometry.is_valid].reset_index(drop=True)
    #gdf = gdf.fillna(gdf.dtypes.replace({"float64": 0.0, "object": "None"}))

    # One version that's a df
    columns_to_drop = ["lower_case_project_title", "lower_case_project_description"]
    #df2 = df2.drop(columns=columns_to_drop + ["geometry"])
    df2 = df2.fillna(df.dtypes.replace({"float64": 0.0, "object": "None"}))

   

    return df2

In [17]:
#ripping amanda's code for find_keywords



In [10]:
# ripping Amanda's code from LRTP project list 



In [17]:
# ripping Amanda's code from LRTP project list 

transit_proj = filter_projects(
    my_df,
    [
        "project_title",
        "project_description",
    ],
    transit_keywords,
    False,
)

/tmp/ipykernel_159/1913973701.py:9: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df[i]


In [19]:
transit_proj.shape

(1729, 13)

In [20]:
#exported data to GCS as CSV
transit_proj.to_csv('gs://calitp-analytics-data/data-analyses/local_transit_projects/local_tranit_project_list.csv')

## Now we have a smaller dataframe

In [2]:
projects = pd.read_csv('gs://calitp-analytics-data/data-analyses/local_transit_projects/local_tranit_project_list.csv')

In [8]:
display(projects.shape)
display(projects.columns)
display(projects.head(3))

(1731, 14)

Index(['Unnamed: 0', 'project_title', 'lead_agency', 'project_year',
       'project_description', 'total_project_cost', 'city', 'county',
       'data_source', 'notes', 'lower_case_project_title',
       'lower_case_project_description',
       'lower_case_project_title_keyword_search',
       'lower_case_project_description_keyword_search'],
      dtype='object')

,Unnamed: 0,project_title,lead_agency,project_year,project_description,total_project_cost,city,county,data_source,notes,lower_case_project_title,lower_case_project_description,lower_case_project_title_keyword_search,lower_case_project_description_keyword_search
0,0,Citywide Sidewalk Improvement Program,Ambag,None,Construct New Sidewalk Per Ada Transition Plan,6000000.0,None,Santa Cruz,Ambag Lrtp,None,citywide sidewalk improvement program,construct new sidewalk per ada transition plan,keyword not found,transit
1,1,Ada Transition Program,Ambag,None,"City-Wide Sidewalk, Ramp, Intersection, And Bu...",1621000.0,None,Santa Cruz,Ambag Lrtp,None,ada transition program,"citywide sidewalk, ramp, intersection, and bus...",transit,bus
2,2,Class I Bike Path Along Railroad,Ambag,None,Install Class I Bike Path Along Railroad Row,1300000.0,None,Santa Cruz,Ambag Lrtp,None,class i bike path along railroad,install class i bike path along railroad row,railroad,railroad


In [11]:
projects.dtypes

Unnamed: 0                                       object
project_title                                    object
lead_agency                                      object
project_year                                     object
project_description                              object
total_project_cost                               object
city                                             object
county                                           object
data_source                                      object
notes                                            object
lower_case_project_title                         object
lower_case_project_description                   object
lower_case_project_title_keyword_search          object
lower_case_project_description_keyword_search    object
dtype: object

In [14]:
#test to convert dtype of total_project_cost to int64
projects['total_project_cost'] = projects['total_project_cost'].astype('float')

ValueError: could not convert string to float: 'Project Type: Bicycle & Pedestrian,  Status: Programmed ,  Fund Estimate: $ 2.4 Million,  Fund Source: Congestion Mitigation And Air Quality Program, Local Agency & Active Transportation Program Funds'

In [12]:
#what are the top 10 projects with the highest project cost
projects.sort_values(by='total_project_cost', ascending=False).head(10)

,Unnamed: 0,project_title,lead_agency,project_year,project_description,total_project_cost,city,county,data_source,notes,lower_case_project_title,lower_case_project_description,lower_case_project_title_keyword_search,lower_case_project_description_keyword_search
35,Project,2400000.0,None,None,Bcag Lrtp,"Project Type: Bicycle & Pedestrian, Status: P...",sr 99 bikeway phase 4 improvements,business lane along the east side of sr 99 cor...,NaN,NaN,NaN,NaN,NaN,NaN
560,558,Roseville Mtce Station,Sacog,2036-2040,"Rebuild Crewrooms, Offices And Eq Barn",999000.0,None,Pla,Sacog Lrtp,Budget Category: C- Maintenance & Rehabilitati...,roseville mtce station,"rebuild crewrooms, offices and eq barn",station,keyword not found
1328,1326,No Title,Scag,2020,In Western Riverside Co. In The City Of Rivers...,997000.0,None,None,Scag Lrtp,"System: Local Highway, Route #: 0, Route Nam...",no title,in western riverside co in the city of riversi...,keyword not found,station
1583,1581,No Title,Scag,2022,Transit Mobility Management Information Center,994000.0,None,None,Scag Lrtp,"System: Transit, Route #: 0, Route Name: Nan...",no title,transit mobility management information center,keyword not found,transit
1142,1140,No Title,Scag,2020,"Purchase Of Replacement Buses: Up To 12, 40-F...",9934000.0,None,None,Scag Lrtp,"System: Transit, Route #: 0, Route Name: Nan...",no title,"purchase of replacement buses up to 12, 40foo...",keyword not found,bus
979,977,No Title,Scag,2020,Eliminate Hazards At Railroad Grade Crossing A...,992000.0,None,None,Scag Lrtp,"System: Local Highway, Route #: 0, Route Nam...",no title,eliminate hazards at railroad grade crossing a...,keyword not found,railroad
606,604,Us 50 / Rancho Cordova Parkway Interchange,Sacog,2026-2030,About 7 Miles East Of Sacramento Between Sunri...,99162000.0,None,Sac,Sacog Lrtp,"Budget Category: B- Road & Highway Capacity, ...",us 50 / rancho cordova parkway interchange,about 7 miles east of sacramento between sunri...,keyword not found,bus
1218,1216,No Title,Scag,2022,Purchase Of Up To 120 Electric 30' To 35' Buse...,99137000.0,None,None,Scag Lrtp,"System: Transit, Route #: 0, Route Name: Nan...",no title,purchase of up to 120 electric 30' to 35' buse...,keyword not found,bus
772,770,Rapid 709,Sandag,2035,H St Trolley Station To Millennia Via H St Cor...,99000000.0,None,None,Sandag Lrtp,"Category: Transit Leap, Status: Proposed, Aq...",rapid 709,h st trolley station to millennia via h st cor...,keyword not found,station
662,660,Circulator Bus/Microtransit Expansion Vehicles,Sacog,2036-2040,Circulator Bus/Microtransit Expansion Vehicles,9885353.0,None,Sac,Sacog Lrtp,Budget Category: E- Transit Capital (Vehicles)...,circulator bus/microtransit expansion vehicles,circulator bus/microtransit expansion vehicles,bus,bus
